**import ee** // ee is the Python package name for the Google Earth Engine Python API.

This statement loads the client library into the Colab Python process.

After this line, objects such as these become available:

ee.Image
ee.ImageCollection
ee.Geometry
ee.FeatureCollection
ee.Reducer

PROJECT_ID = "india-ghsl-urbanization"

ee.Authenticate() //This verifies who you are.

It initiates an OAuth authentication flow, in which you sign into a Google account and authorize the notebook to make Earth Engine requests using that account.

ee.Initialize(project=Project ID) Initialize this Python session using my authenticated credentials, and route my Earth Engine requests through the Cloud project stored in PROJECT_ID.

print("Earth Engine initialized successfully")

Import Earth Engine.

Internal Python attributes inside the directory start with "_" - Can be ignored.

In [ ]:
import ee
import geemap

dir(ee)

['Algorithms',
 'Any',
 'ApiFunction',
 'Array',
 'Authenticate',
 'Blob',
 'Classifier',
 'Clusterer',
 'Collection',
 'ComputedObject',
 'ConfusionMatrix',
 'CustomFunction',
 'Date',
 'DateRange',
 'Dictionary',
 'EEException',
 'Element',
 'Encodable',
 'ErrorMargin',
 'Feature',
 'FeatureCollection',
 'Filter',
 'Function',
 'Geometry',
 'Hashable',
 'Image',
 'ImageCollection',
 'Initialize',
 'Join',
 'Kernel',
 'List',
 'Model',
 'NO_PROJECT_EXCEPTION',
 'Number',
 'PixelType',
 'Projection',
 'Reducer',
 'Reset',
 'Sequence',
 'Serializer',
 'ServiceAccountCredentials',
 'String',
 'TYPE_CHECKING',
 'Terrain',
 '_AlgorithmsContainer',
 '_DYNAMIC_CLASSES',
 '_HAS_DYNAMIC_ATTRIBUTES',
 '_InitializeDeprecatedAssets',
 '_InitializeGeneratedClasses',
 '_InitializeUnboundMethods',
 '_MakeClass',
 '_Promote',
 '_ResetGeneratedClasses',
 '__annotations__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '_

Establish the connection:

*   Authenticated Google account
*   Earth Engine-enabled Cloud project
*   Earth Engine servers

In [ ]:
help(ee.Authenticate)
help(ee.Initialize)
help(ee.Image)

Help on function Authenticate in module ee:

Authenticate(
    authorization_code: str | None = None,
    quiet: bool | None = None,
    code_verifier: str | None = None,
    auth_mode: str | None = None,
    scopes: collections.abc.Sequence[str] | None = None,
    force: bool = False
) -> bool | None
    Prompts the user to authorize access to Earth Engine via OAuth2.

    Args:
      authorization_code: An optional authorization code.
      quiet: If true, do not require interactive prompts. If false, never supply
        --no-browser. Default is None, which autodetects the --no-browser setting.
      code_verifier: PKCE verifier to prevent auth code stealing.
      auth_mode: The authentication mode. One of:
        "colab" - use the Colab authentication flow;
        "notebook" - send user to notebook authenticator page;
        "gcloud" - use gcloud to obtain credentials;
        "localhost" - runs auth flow in local browser only;
        None - a default mode is chosen based on y

In [ ]:
PROJECT_ID = "india-ghsl-urbanization"

ee.Authenticate()
ee.Initialize(project = PROJECT_ID)

print("Earth Engine initialized successfully")

Earth Engine initialized successfully


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


**Investigation of the collection** https://developers.google.com/earth-engine/datasets/catalog/JRC_GHSL_P2023A_GHS_SMOD_V2-0

https://human-settlement.emergency.copernicus.eu/documents/GHSL_Data_Package_2023.pdf?t=1727170839

**Dataset identification**

The research requires a multitemporal classification of rural and urban
settlements.

The Google Earth Engine Data Catalog identifies the dataset:

GHSL: Degree of Urbanisation 1975-2030 V2.0 (P2023A)

The catalog publishes the following Earth Engine collection ID:

`JRC/GHSL/P2023A/GHS_SMOD_V2-0`

This identifier is copied from the official catalog. It is not constructed
or inferred in the notebook.

In [ ]:
ghsl_2020 = ee.Image("JRC/GHSL/P2023A/GHS_SMOD_V2-0/2020")

print("GHSL bands:",ghsl_2020.bandNames().getInfo())

GHSL bands: ['smod_code']


In [ ]:
image = ee.Image("JRC/GHSL/P2023A/GHS_SMOD_V2-0/2030")
smod = image.select("smod_code")
m = geemap.Map(center=[25, 84], zoom=5)
m.add_layer(image,{},"Degree of Urbanization")
m

Map(center=[25, 84], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tran…

In [ ]:
image = ee.Image("JRC/GHSL/P2023A/GHS_SMOD_V2-0/2030")
smod = image.select("smod_code")
m = geemap.Map(center=[25, 84], zoom=5)
m.add_layer(smod,{},"Degree of Urbanization")
m

Map(center=[25, 84], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tran…

In [ ]:
print(type(image))
'''
This means image is an Earth Engine Image object.

It is a client-side Python reference to a server-side raster. The complete global raster has not been downloaded into Colab.

The object tells Earth Engine:

Use the raster stored at this asset ID when a computation or visualization is requested.
'''

<class 'ee.image.Image'>


'\nThis means image is an Earth Engine Image object.\n\nIt is a client-side Python reference to a server-side raster. The complete global raster has not been downloaded into Colab.\n\nThe object tells Earth Engine:\n\nUse the raster stored at this asset ID when a computation or visualization is requested.\n'

In [ ]:
smod = image.select("smod_code")
band_names_server = image.bandNames()
print(type(band_names_server))
'The catalog defines smod_code as the Degree of Urbanisation classification band. Its documented pixel size is 1,000 metres. This creates a new Earth Engine image consisting only of the smod_code band from the original image. For this particular source image, there is only one documented analytical band. Still, selecting it explicitly makes the notebook clear and protects later code if a source image contains multiple bands.'

<class 'ee.ee_list.List'>


'The catalog defines smod_code as the Degree of Urbanisation classification band. Its documented pixel size is 1,000 metres. This creates a new Earth Engine image consisting only of the smod_code band from the original image. For this particular source image, there is only one documented analytical band. Still, selecting it explicitly makes the notebook clear and protects later code if a source image contains multiple bands.'

In [ ]:
band_names_python = (
    image
    .bandNames()
    .getInfo()
)

print(band_names_python)

['smod_code']


In [ ]:
projection = smod.projection()
print(type(projection))

<class 'ee.projection.Projection'>


In [ ]:
projection_info = projection.getInfo()
print(projection_info)

{'type': 'Projection', 'wkt': 'PROJCS["World_Mollweide", \n  GEOGCS["WGS 84", \n    DATUM["WGS_1984", \n      SPHEROID["WGS 84", 6378137.0, 298.257223563, AUTHORITY["EPSG","7030"]], \n      AUTHORITY["EPSG","6326"]], \n    PRIMEM["Greenwich", 0.0], \n    UNIT["degree", 0.017453292519943295], \n    AXIS["Longitude", EAST], \n    AXIS["Latitude", NORTH]], \n  PROJECTION["Mollweide"], \n  PARAMETER["semi_minor", 6378137.0], \n  PARAMETER["false_easting", 0.0], \n  PARAMETER["false_northing", 0.0], \n  PARAMETER["central_meridian", 0.0], \n  UNIT["m", 1.0], \n  AXIS["Easting", EAST], \n  AXIS["Northing", NORTH]]', 'transform': [1000, 0, -18041000, 0, -1000, 9000000]}


In [ ]:
nominal_scale = (
    smod
    .projection()
    .nominalScale()
)

print(type(nominal_scale))

<class 'ee.ee_number.Number'>


In [ ]:
nominal_scale_metres = (
    nominal_scale
    .getInfo()
)

print("Nominal pixel scale:", nominal_scale_metres, "metres")

Nominal pixel scale: 1000 metres


In [ ]:
property_names = (
    image
    .propertyNames()
    .getInfo()
)

print(property_names)

['system:time_start', 'system:footprint', 'smod_code_class_names', 'system:time_end', 'system:version', 'smod_code_class_values', 'system:id', 'smod_code_class_palette', 'system:asset_size', 'system:index', 'system:bands', 'system:band_names']


In [ ]:
image_info = image.getInfo()

print(image_info.keys())

dict_keys(['type', 'bands', 'version', 'id', 'properties'])


So the raster uses the World Mollweide projected coordinate system.

Why Mollweide?

Mollweide is an equal-area projection. That matters for this project because we will compare areas occupied by settlement classes across India and through time.

In an equal-area projection:

Equal-sized pixels represent equal ground area consistently across the map.

That makes area-based calculations more defensible than using latitude-longitude degrees directly.

The official GHSL source also lists Mollweide as a coordinate system for GHS-SMOD and identifies the product as a 1 km settlement-classification grid.

In [ ]:
print("Python object type:")
print(type(image))

print("\nBands:")
print(image.bandNames().getInfo())

print("\nProjection:")
print(smod.projection().getInfo())

print("\nNominal scale in metres:")
print(
    smod
    .projection()
    .nominalScale()
    .getInfo()
)

print("\nImage metadata property names:")
print(image.propertyNames().getInfo())

Python object type:
<class 'ee.image.Image'>

Bands:
['smod_code']

Projection:
{'type': 'Projection', 'wkt': 'PROJCS["World_Mollweide", \n  GEOGCS["WGS 84", \n    DATUM["WGS_1984", \n      SPHEROID["WGS 84", 6378137.0, 298.257223563, AUTHORITY["EPSG","7030"]], \n      AUTHORITY["EPSG","6326"]], \n    PRIMEM["Greenwich", 0.0], \n    UNIT["degree", 0.017453292519943295], \n    AXIS["Longitude", EAST], \n    AXIS["Latitude", NORTH]], \n  PROJECTION["Mollweide"], \n  PARAMETER["semi_minor", 6378137.0], \n  PARAMETER["false_easting", 0.0], \n  PARAMETER["false_northing", 0.0], \n  PARAMETER["central_meridian", 0.0], \n  UNIT["m", 1.0], \n  AXIS["Easting", EAST], \n  AXIS["Northing", NORTH]]', 'transform': [1000, 0, -18041000, 0, -1000, 9000000]}

Nominal scale in metres:
1000

Image metadata property names:
['system:time_start', 'system:footprint', 'smod_code_class_names', 'system:time_end', 'system:version', 'smod_code_class_values', 'system:id', 'smod_code_class_palette', 'system:asset_s

In [ ]:
class_values = (
    image
    .get("smod_code_class_values")
    .getInfo()
)

class_names = (
    image
    .get("smod_code_class_names")
    .getInfo()
)

class_palette = (
    image
    .get("smod_code_class_palette")
    .getInfo()
)

print("Class values:")
print(class_values)

print("\nClass names:")
print(class_names)

print("\nClass palette:")
print(class_palette)

Class values:
[10, 11, 12, 13, 21, 22, 23, 30]

Class names:
['Water', 'Very low density rural', 'Low density rural', 'Rural cluster', 'Suburban or peri-urban', 'Semi-dense urban cluster', 'Dense urban cluster', 'Urban centre']

Class palette:
['7ab6f5', 'cdf57a', 'abcd66', '375623', 'ffff00', 'a87000', '732600', 'ff0000']


In [ ]:
print("Number of values:", len(class_values))
print("Number of names:", len(class_names))
print("Number of colours:", len(class_palette))

Number of values: 8
Number of names: 8
Number of colours: 8


In [ ]:
legend_rows = list(zip(class_values, class_names, class_palette))

for value, name, colour in legend_rows:
    print(value,"->",name,"->",colour)

10 -> Water -> 7ab6f5
11 -> Very low density rural -> cdf57a
12 -> Low density rural -> abcd66
13 -> Rural cluster -> 375623
21 -> Suburban or peri-urban -> ffff00
22 -> Semi-dense urban cluster -> a87000
23 -> Dense urban cluster -> 732600
30 -> Urban centre -> ff0000


In [ ]:
start_date = ee.Date(image.get("system:time_start"))

end_date = ee.Date(image.get("system:time_end"))

print("Start date:", start_date.format("YYYY-MM-dd").getInfo())

print("End date:", end_date.format("YYYY-MM-dd").getInfo())

Start date: 2030-01-01
End date: 2035-01-01


In [ ]:
print("System ID:", image.get("system:id").getInfo())

print("System index:", image.get("system:index").getInfo())

print("Asset size in bytes:", image.get("system:asset_size").getInfo())

System ID: JRC/GHSL/P2023A/GHS_SMOD_V2-0/2030
System index: 2030
Asset size in bytes: 13117750


Object type:
Earth Engine Image

Spatial model:
Global raster grid

Projection:
World Mollweide

Native grid spacing:
1,000 metres

Variables:
One band, smod_code

Variable type:
Categorical settlement classification

Temporal identity:
2030 epoch

Metadata:
Class values, labels, palette, time interval,
asset identity, asset size, and footprint

Until now we have done
Earth Engine connection             Complete
Dataset identification              Complete
Image and band validation           Complete
Projection and grid inspection      Complete
Temporal/asset metadata inspection  Complete
India boundary selection            Next
Historical epoch selection          Next
Small test export                    Next
India historical exports            After test validation
HDFS and Spark ingestion             Immediately after exports

In [ ]:
class_values = (
    image
    .get("smod_code_class_values")
    .getInfo()
)

class_names = (
    image
    .get("smod_code_class_names")
    .getInfo()
)

class_palette = (
    image
    .get("smod_code_class_palette")
    .getInfo()
)

for value, name, colour in zip(
    class_values,
    class_names,
    class_palette
):
    print(
        value,
        "->",
        name,
        "->",
        colour
    )

10 -> Water -> 7ab6f5
11 -> Very low density rural -> cdf57a
12 -> Low density rural -> abcd66
13 -> Rural cluster -> 375623
21 -> Suburban or peri-urban -> ffff00
22 -> Semi-dense urban cluster -> a87000
23 -> Dense urban cluster -> 732600
30 -> Urban centre -> ff0000


Create Year 2020 Report.

In [ ]:
GHSL_COLLECTION_ID = (
    "JRC/GHSL/P2023A/GHS_SMOD_V2-0"
)

SELECTED_YEAR = 2020

image_2020 = ee.Image(
    f"{GHSL_COLLECTION_ID}/{SELECTED_YEAR}"
)

smod_2020 = image_2020.select(
    "smod_code"
)

GHSL is global. We do not need the entire world for the project, so the export requires a region.

One available public boundary collection in Earth Engine is: USDOS/LSIB_SIMPLE/2017

It is an Earth Engine FeatureCollection of simplified international-boundary polygons. The catalog notes that it excludes medium and smaller islands and reflects U.S. government policies on boundaries and disputed areas, so this boundary choice must be documented as an implementation input rather than treated as India’s official administrative boundary.

First inspect its schema:

In [ ]:
COUNTRY_BOUNDARY_ID = (
    "USDOS/LSIB_SIMPLE/2017"
)

countries = ee.FeatureCollection(
    COUNTRY_BOUNDARY_ID
)

print(
    countries.first().propertyNames().getInfo()
)

['wld_rgn', 'country_na', 'abbreviati', 'system:index', 'country_co']


Country_na is the column to inspect for India.

Next print needed? Not really!

In [ ]:
print(
    countries.first().toDictionary().getInfo()
)

{'abbreviati': '', 'country_co': 'CD', 'country_na': 'Chad', 'wld_rgn': 'Africa'}


In [ ]:
india = countries.filter(
    ee.Filter.eq(
        "country_na",
        "India"
    )
)

print(
    "Number of matching features:",
    india.size().getInfo()
)

Number of matching features: 1


In [ ]:
india_outline = india.style(
    color="000000",
    fillColor="00000000",
    width=2
)

m.add_layer(
    india_outline,
    {},
    "India boundary"
)

m

Map(center=[25, 84], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tran…

In [ ]:
GHSL_COLLECTION_ID = (
    "JRC/GHSL/P2023A/GHS_SMOD_V2-0"
)

SELECTED_YEAR = 2020

image_2020 = ee.Image(
    f"{GHSL_COLLECTION_ID}/{SELECTED_YEAR}"
)

smod_2020 = image_2020.select(
    "smod_code"
)

print(
    "Loaded asset:",
    image_2020.get("system:id").getInfo()
)

print(
    "Available bands:",
    image_2020.bandNames().getInfo()
)

Loaded asset: JRC/GHSL/P2023A/GHS_SMOD_V2-0/2020
Available bands: ['smod_code']


In [ ]:
source_projection = (
    smod_2020
    .projection()
    .getInfo()
)

source_crs = source_projection["wkt"]

source_transform = source_projection[
    "transform"
]

print("Source CRS:")
print(source_crs)

print("\nSource transform:")
print(source_transform)

Source CRS:
PROJCS["World_Mollweide", 
  GEOGCS["WGS 84", 
    DATUM["WGS_1984", 
      SPHEROID["WGS 84", 6378137.0, 298.257223563, AUTHORITY["EPSG","7030"]], 
      AUTHORITY["EPSG","6326"]], 
    PRIMEM["Greenwich", 0.0], 
    UNIT["degree", 0.017453292519943295], 
    AXIS["Longitude", EAST], 
    AXIS["Latitude", NORTH]], 
  PROJECTION["Mollweide"], 
  PARAMETER["semi_minor", 6378137.0], 
  PARAMETER["false_easting", 0.0], 
  PARAMETER["false_northing", 0.0], 
  PARAMETER["central_meridian", 0.0], 
  UNIT["m", 1.0], 
  AXIS["Easting", EAST], 
  AXIS["Northing", NORTH]]

Source transform:
[1000, 0, -18041000, 0, -1000, 9000000]


In [ ]:
export_task_2020 = ee.batch.Export.image.toDrive(
    image=smod_2020,
    description="GHSL_SMOD_India_2020_Test",
    folder="GHSL_India",
    fileNamePrefix="ghsl_smod_india_2020",
    region=india.geometry(),
    crs=source_crs,
    crsTransform=source_transform,
    maxPixels=10_000_000_000,
    fileFormat="GeoTIFF"
)

export_task_2020.start()

print(export_task_2020.status())

{'state': 'READY', 'description': 'GHSL_SMOD_India_2020_Test', 'priority': 100, 'creation_timestamp_ms': 1789999971702, 'update_timestamp_ms': 1789999971702, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'THSRKNUIEKESJRD4HVSGFNTL', 'name': 'projects/india-ghsl-urbanization/operations/THSRKNUIEKESJRD4HVSGFNTL'}


In [ ]:
status = export_task_2020.status()

print("Task state:", status["state"])

print(
    "Error message:",
    status.get("error_message", "No error reported")
)

Task state: READY
Error message: No error reported


In [ ]:
status = export_task_2020.status()

print("Task state:", status["state"])

print(
    "Error message:",
    status.get("error_message", "No error reported")
)

Task state: READY
Error message: No error reported


In [ ]:
status = export_task_2020.status()

print("Task state:", status["state"])

print(
    "Error message:",
    status.get("error_message", "No error reported")
)

Task state: READY
Error message: No error reported


In [ ]:
status = export_task_2020.status()

print("Task state:", status["state"])

print(
    "Error message:",
    status.get("error_message", "No error reported")
)

Task state: COMPLETED
Error message: No error reported


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

export_folder = (
    "/content/drive/MyDrive/GHSL_India"
)

for file_name in os.listdir(export_folder):
    file_path = os.path.join(
        export_folder,
        file_name
    )

    file_size_mb = (
        os.path.getsize(file_path)
        / (1024 * 1024)
    )

    print(
        file_name,
        "->",
        round(file_size_mb, 2),
        "MB"
    )

ghsl_smod_india_2020.tif -> 0.9 MB


Before transferring the file into HDFS, we must confirm that the exported file preserved the source raster’s structure.

rasterio.open() opens the GeoTIFF as a raster dataset reader. Rasterio exposes the coordinate reference system through src.crs and the pixel-to-coordinate mapping through src.transform.

What each property tells us
src.driver: confirms the file format, normally GTiff.
src.count: number of raster bands, expected 1.
src.width, src.height: number of pixel columns and rows.
src.dtypes: storage datatype for the raster values.
src.crs: exported coordinate reference system.
src.transform: exported pixel size and grid origin.
src.nodata: value representing pixels without valid data.
src.bounds: projected geographic extent of the exported raster.

In [ ]:
import rasterio

tif_path = (
    "/content/drive/MyDrive/GHSL_India/"
    "ghsl_smod_india_2020.tif"
)

with rasterio.open(tif_path) as src:
    print("Driver:", src.driver)
    print("Band count:", src.count)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Data type:", src.dtypes)
    print("CRS:", src.crs)
    print("Transform:", src.transform)
    print("No-data value:", src.nodata)
    print("Bounds:", src.bounds)

Driver: GTiff
Band count: 1
Width: 2907
Height: 3448
Data type: ('int16',)
CRS: ESRI:54009
Transform: | 1000.00, 0.00, 6467000.00|
| 0.00,-1000.00, 4282000.00|
| 0.00, 0.00, 1.00|
No-data value: None
Bounds: BoundingBox(left=6467000.0, bottom=834000.0, right=9374000.0, top=4282000.0)


No-data value: None  We need to discover how pixels outside India were represented during export before sending the file to Spark.

In [ ]:
import numpy as np
import rasterio

with rasterio.open(tif_path) as src:
    data = src.read(1)
    valid_mask = src.read_masks(1)

print(
    "Unique pixel values:",
    np.unique(data)
)

print(
    "Unique mask values:",
    np.unique(valid_mask)
)

Unique pixel values: [ 0 10 11 12 13 21 22 23 30]
Unique mask values: [255]


Good catch. The export reveals an important data-quality issue:

Valid GHSL classes are 10, 11, 12, 13, 21, 22, 23, 30.
0 is not a valid GHSL urbanisation class.
The mask contains only 255, so Rasterio currently considers every pixel valid.
src.nodata is None, so the exported file does not formally identify 0 as missing data.

In [ ]:
import numpy as np
import rasterio

with rasterio.open(tif_path) as src:
    data = src.read(1)

values, counts = np.unique(
    data,
    return_counts=True
)

for value, count in zip(values, counts):
    percentage = (
        count / data.size
    ) * 100

    print(
        f"Value {value:>3}: "
        f"{count:>10,} pixels "
        f"({percentage:.2f}%)"
    )

Value   0:  6,844,740 pixels (68.29%)
Value  10:     36,010 pixels (0.36%)
Value  11:  1,603,493 pixels (16.00%)
Value  12:    756,527 pixels (7.55%)
Value  13:    137,077 pixels (1.37%)
Value  21:    478,978 pixels (4.78%)
Value  22:     41,063 pixels (0.41%)
Value  23:     50,207 pixels (0.50%)
Value  30:     75,241 pixels (0.75%)


We preserve: ghsl_smod_india_2020.tif as the untouched raw export. We create: ghsl_smod_india_2020_clean.tif as the curated Spark input.

Raw layer → Original Earth Engine export

Curated layer → Validated file with explicit NoData metadata

In [ ]:
import rasterio

clean_tif_path = (
    "/content/drive/MyDrive/GHSL_India/"
    "ghsl_smod_india_2020_clean.tif"
)

with rasterio.open(tif_path) as src:
    profile = src.profile.copy()

    profile.update(
        nodata=0,
        compress="lzw"
    )

    with rasterio.open(
        clean_tif_path,
        "w",
        **profile
    ) as dst:
        dst.write(src.read(1), 1)

print("Created:", clean_tif_path)


Created: /content/drive/MyDrive/GHSL_India/ghsl_smod_india_2020_clean.tif


In [ ]:
import numpy as np
import rasterio

with rasterio.open(clean_tif_path) as src:
    clean_data = src.read(1)
    clean_mask = src.read_masks(1)

    print("No-data value:", src.nodata)
    print("Unique stored values:", np.unique(clean_data))
    print("Unique mask values:", np.unique(clean_mask))

    print(
        "Invalid pixel count:",
        np.count_nonzero(clean_mask == 0)
    )

    print(
        "Valid pixel count:",
        np.count_nonzero(clean_mask == 255)
    )

No-data value: 0.0
Unique stored values: [ 0 10 11 12 13 21 22 23 30]
Unique mask values: [  0 255]
Invalid pixel count: 6844740
Valid pixel count: 3178596


Next step only: turn the checks into automated assertions Why write assertions?

Until now, we manually inspected outputs. An assertion converts an expected property into an executable data-quality rule.

assert src.count == 1

Stop execution if the exported file does not contain exactly one raster band.
assert src.nodata == 0
means-  Stop execution unless zero is formally defined as NoData.

In [ ]:
import numpy as np
import rasterio

expected_class_codes = {
    10, 11, 12, 13,
    21, 22, 23, 30
}

with rasterio.open(clean_tif_path) as src:
    data = src.read(1)
    mask = src.read_masks(1)

    stored_values = set(
        np.unique(data).tolist()
    )

    valid_values = set(
        np.unique(data[mask == 255]).tolist()
    )

    assert src.count == 1
    assert src.nodata == 0
    assert stored_values == expected_class_codes | {0}
    assert valid_values == expected_class_codes
    assert np.count_nonzero(mask == 0) == 6_844_740
    assert np.count_nonzero(mask == 255) == 3_178_596

print("All GeoTIFF validation checks passed.")


All GeoTIFF validation checks passed.
